In [1]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset , DataLoader 
import torch.nn as nn
import torch.nn.functional as F
import math

In [2]:
pd.set_option('display.max_colwidth', None)

In [3]:
file_path = './G2P_v2.csv'

In [4]:
data = pd.read_csv(file_path, low_memory=False) 

In [5]:
data = data.drop(columns=['Homograph Grapheme','Homograph Phoneme','Source','Source ID','Mapped Phoneme','Mapped Homograph Phoneme'])

In [6]:
unwanted_chars = [
    '\n', '\u200b','\u200c','\u200e','\u200f','\ufeff','\xa0',
    '!', '"', '#', '&', "'", '(', ')', '+', ',', '-', '.', '/', 
    ':',';','<','=','?','@','_','–','“','”','…','«','»','٪','ـ',
    '0','1','2','3','4','5','6','7','8','9',
    '۰','۱','۲','۳','۴','۵','۶','۷','۸','۹', '٠','١','٢','٣','٤','٥','٦','٧','٨','٩',
    '،','؛','؟','٫','٬'
    'ü','ā','š','ʔ','新','最','\xad'
]

In [7]:
pattern = "[" + "".join(unwanted_chars) + "]"
data['Grapheme'] = data['Grapheme'].apply(lambda x: re.sub(pattern, '', x))

In [8]:
unwanted_phoneme_char = [
    '\n' , '_','-'
]

In [9]:
data =data.dropna()
# data

In [10]:
pattern_p =  "[" + "".join(unwanted_phoneme_char) + "]"
data['Phoneme'] = data['Phoneme'].apply(lambda x : re.sub(pattern_p,'',x))

In [11]:
mapping = {

    'A': 'آ', 'B': 'بی', 'C': 'سی', 'D': 'دی', 'E': 'ای',
    'F': 'اف', 'G': 'جی', 'H': 'اچ', 'I': 'آی', 'J': 'جِی',
    'K': 'کِی', 'L': 'ال', 'M': 'ام', 'N': 'ان', 'O': 'او',
    'P': 'پی', 'Q': 'کیو', 'R': 'آر', 'S': 'اس', 'T': 'تی',
    'U': 'یو', 'V': 'وی', 'W': 'دبلیو', 'X': 'ایکس', 'Y': 'وای', 'Z': 'زد',
    'a': 'آ', 'b': 'بی', 'c': 'سی', 'd': 'دی', 'e': 'ای',
    'f': 'اف', 'g': 'جی', 'h': 'اچ', 'i': 'آی', 'j': 'جِی',
    'k': 'کِی', 'l': 'ال', 'm': 'ام', 'n': 'ان', 'o': 'او',
    'p': 'پی', 'q': 'کیو', 'r': 'آر', 's': 'اس', 't': 'تی',
    'u': 'یو', 'v': 'وی', 'w': 'دبلیو', 'x': 'ایکس', 'y': 'وای', 'z': 'زد'
}


In [12]:
def replace_upper(match):
    return mapping[match.group(0)]

In [13]:
data['Grapheme'] = data['Grapheme'].apply(lambda x: re.sub(r'[A-Za-z]', replace_upper, x))

In [14]:
# data

In [15]:
grapheme = data['Grapheme'].to_list()
phoneme = data['Phoneme'].to_list()

In [16]:
grapheme_chars = set()
for text in grapheme :
    grapheme_chars.update(list(text))
grapheme_chars = sorted(list(grapheme_chars))
print(grapheme_chars)

[' ', 'ء', 'آ', 'أ', 'ؤ', 'إ', 'ئ', 'ا', 'ب', 'ة', 'ت', 'ث', 'ج', 'ح', 'خ', 'د', 'ذ', 'ر', 'ز', 'س', 'ش', 'ص', 'ض', 'ط', 'ظ', 'ع', 'غ', 'ف', 'ق', 'ك', 'ل', 'م', 'ن', 'ه', 'و', 'ى', 'ي', 'ً', 'ٌ', 'َ', 'ُ', 'ِ', 'ّ', 'ْ', 'ٔ', 'پ', 'چ', 'ژ', 'ک', 'گ', 'ۀ', 'ی', 'ے', 'ە', 'ﭘ', 'ﮐ', 'ﮔ', 'ﯽ', 'ﯾ', 'ﯿ', 'ﺍ', 'ﺎ', 'ﺑ', 'ﺒ', 'ﺖ', 'ﺘ', 'ﺧ', 'ﺩ', 'ﺪ', 'ﺭ', 'ﺮ', 'ﺱ', 'ﺴ', 'ﺷ', 'ﺸ', 'ﻀ', 'ﻋ', 'ﻌ', 'ﻟ', 'ﻡ', 'ﻢ', 'ﻤ', 'ﻥ', 'ﻧ', 'ﻪ', 'ﻭ', 'ﻮ']


In [17]:
data

,Grapheme,Phoneme
0,روی دیوار ننویسید,ruye divAr nanevisid
1,روی زمین بخوابید,ruye zamin bexAbid
2,خاک را زیر و رو کنید,xak rA zir ?o ru konid
3,انسان باید با خودش رو راست باشد,?ensAn bAyad bA xodaS ru rAst bASad
4,دست روی دلم نذار که خون است,dast ruye delam nzAr ke xun ?ast
...,...,...
528927,و همکاری صمیمانه ایشان نابینایان,va hamkAriye samimAneye ?iSAn ?e nAbinAyAn
528928,میکنم پیش از شروع تشریفات اقدسیه و مدیریت محترم آن,mikonam piS ?az Soru?ye taSrifAte ?aqdasiyye va modiriyyate mohtaram ?An
528929,صمیمانه ایشان همواره شامل حال انجمن نابینایان ایران را به نمایش,samimAne ?iSAn hamvAre SAmele hAle ?anjomane nAbinayAne ?irAn rA be namAyeS
528930,بار دیگر از همکاریهای مهربانی تشکر ویژهای داشته باشم مهربانی و همکاری,bAre digar ?az hamkArihAye mehrabAni taSakkore vizhei dASte bASam mehrabAni va hamkAri


In [18]:
data['Grapheme_tokens'] = data['Grapheme'].apply(list)

In [19]:
data['Phoneme_tokens'] = data['Phoneme'].apply(list)

In [20]:
def tokenize_phoneme(phoneme):
    chars = list(phoneme)
    return ['<sos>'] + chars + ['<eos>']

In [21]:
data['Phoneme_tokens'] = data['Phoneme'].apply(tokenize_phoneme)
data.to_csv('PersianG2P_preprocessed.csv', encoding="utf-8",index=False)

In [22]:
all_chars_dict = set (ch for toks in data['Grapheme_tokens']for ch in toks)
grapheme_dict = {'<pad>': 0}
for i , ch in enumerate(sorted(all_chars_dict),start=1):
    grapheme_dict[ch] = i


In [23]:
# grapheme_dict

In [24]:
all_phone_dict = set(ph for phons in data['Phoneme_tokens']for ph in phons)
phoneme_dict = {'<pad>' : 0}
for i,ph in enumerate(sorted(all_phone_dict),start=1):
    phoneme_dict[ph] = i

In [25]:
# phoneme_dict

In [26]:
def grapheme_toID(tokens , graph_dict):
    return[graph_dict[t] for t in tokens]

In [27]:
def phoneme_toID(phones , phone_dict):
    return[phone_dict[p] for p in phones]

In [28]:
data['Grapheme_tokens'] = data['Grapheme_tokens'].apply(lambda x : grapheme_toID(x , grapheme_dict))
data['Phoneme_tokens'] = data['Phoneme_tokens'].apply(lambda x : phoneme_toID(x , phoneme_dict))

In [29]:
data['grapheme_len'] = data['Grapheme_tokens'].apply(len)

In [30]:
data['phoneme_len'] = data['Phoneme_tokens'].apply(len)

In [31]:
data_sorted = data.sort_values(by='phoneme_len', ascending=False)

In [32]:
data = data[data['phoneme_len'] <= 2 * data['grapheme_len']]

In [33]:
max_p = data['phoneme_len'].max()
max_g = data['grapheme_len'].max()
print(max_p)
print(max_g)

250
198


In [34]:
PAD  = 0
def pad(seq, max_len):
    return seq + [PAD] * (max_len - len(seq))

In [35]:
data = data.copy()
data['Grapheme_tokens'] = data['Grapheme_tokens'].apply(lambda x : pad(x,max_g))
data['Phoneme_tokens'] = data['Phoneme_tokens'].apply(lambda x : pad(x,max_p))

In [36]:
data['grapheme_len'] = data['Grapheme_tokens'].apply(len)
data['phoneme_len'] = data['Phoneme_tokens'].apply(len)

In [37]:
data.to_csv('len.csv',encoding='utf-8', index=False)

In [38]:
# data

In [39]:
data = data[['Grapheme_tokens','Phoneme_tokens']]
X = data['Grapheme_tokens']
y = data['Phoneme_tokens']

X_train , X_test , y_train , y_test = train_test_split(X,y,test_size=0.2,random_state=42,shuffle=True)

In [40]:
class G2PDataset (Dataset):
    def __init__(self , X ,y):
        self.X = X.tolist()
        self.y = y.tolist()
        
    def __len__(self):
        
        return len(self.X)
        
    def __getitem__(self , idx):
        x = torch.tensor(self.X[idx],dtype = torch.long)
        y = torch.tensor(self.y[idx],dtype = torch.long)
        x_mask = (x != 0 ).long()
        y_mask = (y != 0 ).long()
        
        return {
            'grapheme': x,
            'phoneme': y,
        }


In [41]:
batch_size = 64

In [42]:
train_dataset = G2PDataset(X_train,y_train)
test_dataset = G2PDataset(X_test,y_test)

In [43]:
train_loader = DataLoader(train_dataset , batch_size= batch_size , shuffle= True)
test_loader = DataLoader(test_dataset , batch_size=batch_size ,shuffle=False)

<h4>MODEL ARCHITECTURE</h4>

In [44]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)

        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div)
        pe[:, 1::2] = torch.cos(position * div)

        pe = pe.unsqueeze(0)  
        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return x


In [45]:
class G2PTransformer(nn.Module):
    def __init__(self, input_dim, output_dim,
                 d_model=256, nhead=8,
                 num_encoder_layers=3, num_decoder_layers=3,
                 dim_feedforward=512, dropout=0.1, pad_idx=0):
        super().__init__()

        self.embedding_src = nn.Embedding(input_dim, d_model, padding_idx=pad_idx)
        self.embedding_tgt = nn.Embedding(output_dim, d_model, padding_idx=pad_idx)
        self.pos_encoder = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_encoder_layers)

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_decoder_layers)

        self.fc_out = nn.Linear(d_model, output_dim)

    def forward(self, src, tgt):
        # ماسک‌های padding
        src_mask = (src == 0)
        tgt_mask = (tgt == 0)
        look_ahead_mask = create_look_ahead_mask(tgt.size(1)).to(src.device)

        # امبدینگ + پوزیشنال
        src_emb = self.pos_encoder(self.embedding_src(src))
        tgt_emb = self.pos_encoder(self.embedding_tgt(tgt))

        # انکودر
        memory = self.encoder(src_emb, src_key_padding_mask=src_mask)

        # دیکودر
        out = self.decoder(
            tgt_emb, memory,
            tgt_mask=look_ahead_mask,
            tgt_key_padding_mask=tgt_mask,
            memory_key_padding_mask=src_mask
        )

        return self.fc_out(out)


In [46]:
pad_idx = 0
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = G2PTransformer(
    input_dim=len(grapheme_dict),
    output_dim=len(phoneme_dict),
    d_model=256,
    nhead=8,
    num_encoder_layers=6,
    num_decoder_layers=6,
    pad_idx=pad_idx
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


In [47]:
model.to(device)

G2PTransformer(
  (embedding_src): Embedding(88, 256, padding_idx=0)
  (embedding_tgt): Embedding(33, 256, padding_idx=0)
  (pos_encoder): PositionalEncoding()
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-5): 6 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=512, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=512, out_features=256, bias=True)
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (decoder): TransformerDecoder(
    (layers): ModuleList(
      (0-5): 6 x TransformerDecoderLayer(
        (self_attn): MultiheadAtte

In [48]:
def create_src_padding_mask(src, pad_idx=0):
    return (src == pad_idx)  

def create_tgt_padding_mask(tgt, pad_idx=0):
    return (tgt == pad_idx)

def create_look_ahead_mask(size):
    mask = torch.triu(torch.ones(size, size), diagonal=1).bool()
    return mask 


In [49]:
def train_step(model, src, tgt, criterion, optimizer, pad_idx=0):
    model.train()

    tgt_input = tgt[:, :-1]
    tgt_output = tgt[:, 1:]

    # masks
    src_pad_mask = create_src_padding_mask(src, pad_idx)
    tgt_pad_mask = create_tgt_padding_mask(tgt_input, pad_idx)
    seq_len = tgt_input.size(1)
    look_ahead = create_look_ahead_mask(seq_len).to(src.device)

    # forward
    logits = model(
        src,
        tgt_input,
        src_key_padding_mask=src_pad_mask,
        tgt_key_padding_mask=tgt_pad_mask,
        tgt_mask=look_ahead
    )

    logits = logits.reshape(-1, logits.size(-1))
    tgt_output = tgt_output.reshape(-1)

    loss = criterion(logits, tgt_output)


    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()


In [ ]:
num_epochs = 10

for epoch in range(num_epochs):
    total_loss = 0.0

    for batch_idx, batch in enumerate(train_loader):
        # ورودی‌ها
        src = batch['grapheme'].to(device)
        tgt = batch['phoneme'].to(device)

        # یک مرحله آموزش
        optimizer.zero_grad()
        output = model(src, tgt)  # ماسک‌ها داخل forward ساخته می‌شوند

        # CrossEntropyLoss نیاز به (N, C) و target (N,) دارد
        output_reshaped = output.view(-1, output.size(-1))
        tgt_reshaped = tgt.view(-1)
        loss = criterion(output_reshaped, tgt_reshaped)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # لاگ هر 100 استپ
        if batch_idx % 100 == 0:
            print(f"Epoch {epoch+1} | Step {batch_idx} | Loss = {loss.item():.4f}")

    # میانگین لاگ هر اپوک
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} Finished | Avg Loss = {avg_loss:.4f}")


Epoch 1 | Step 0 | Loss = 3.5182
Epoch 1 | Step 100 | Loss = 0.0357
Epoch 1 | Step 200 | Loss = 0.0136
Epoch 1 | Step 300 | Loss = 0.0078
Epoch 1 | Step 400 | Loss = 0.0054
Epoch 1 | Step 500 | Loss = 0.0040
Epoch 1 | Step 600 | Loss = 0.0031
